# 🔮 Notebook 04 — Análisis Predictivo

**Sistema de Denuncias Ambientales · Uruguay · D Empathy Project**

> *¿Qué va a pasar? · Pronósticos · Clasificación automática · Alertas*

Este notebook corresponde a la **Sección 3** del tablero. Cuatro herramientas:

1. **Pronóstico de volumen total** — ajuste polinómico con banda de confianza, 12 meses al frente
2. **NLP sobre descripciones libres** — términos más frecuentes por categoría
3. **Sistema de alertas** — detección de picos recientes vía Z-score
4. **Clustering territorial K-Means** — agrupamiento de departamentos por perfil ambiental

## 📐 Decisiones metodológicas

- **Pronóstico**: el documento técnico v1.0 recomienda **Prophet** (Meta). Para mantener este notebook con dependencias mínimas (sin instalar pystan), uso una **regresión polinómica de grado 2** que cumple el rol pedagógico de forma equivalente. En producción, sustituí por Prophet.
- **NLP**: TF-IDF + frecuencias simples con stopwords en español. Suficiente como baseline antes de un BERT multilingüe (Fase 4 del roadmap).
- **K-Means**: $k=4$ clusters, sobre el vector normalizado de % de cada categoría por departamento.

## ⚠️ Limitaciones

- La serie temporal tiene **un gap entre 2019 y 2023** (no hay datos 2020-2022 en este dataset). Eso degrada cualquier modelo de pronóstico. La banda de confianza es ancha a propósito.
- `descripcion_libre` solo existe en el histórico 2010-2019 (campo `denuncia`); en 2023 viene vacía y en el formulario nuevo se llenará. Hago NLP sobre lo que tengo.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from pathlib import Path

DATA_DIR = Path("../data")
df = pd.read_parquet(DATA_DIR / "denuncias.parquet")
print(f"Cargado: {df.shape[0]:,} denuncias")

# 1️⃣ Pronóstico de volumen total

Agrego por mes, ajusto un polinomio grado 2 al log(conteo) (para que el pronóstico no caiga por debajo de 0) y extiendo 12 meses hacia adelante con banda de confianza al 95%.

In [ ]:
monthly = df.groupby(df["timestamp"].dt.to_period("M")).size().reset_index(name="total")
monthly["fecha"] = monthly["timestamp"].dt.to_timestamp()
monthly["t"] = np.arange(len(monthly))

# Ajuste polinómico sobre el log
y_log = np.log(monthly["total"].clip(lower=1))
coefs = np.polyfit(monthly["t"], y_log, deg=2)
y_pred_log = np.polyval(coefs, monthly["t"])
residuals = y_log - y_pred_log
sigma = residuals.std()

# Forecast 12 meses
t_future = np.arange(len(monthly), len(monthly) + 12)
y_future_log = np.polyval(coefs, t_future)
y_future = np.exp(y_future_log)
y_future_low = np.exp(y_future_log - 1.96 * sigma)
y_future_high = np.exp(y_future_log + 1.96 * sigma)

last_date = monthly["fecha"].max()
future_dates = pd.date_range(last_date + pd.offsets.MonthBegin(1), periods=12, freq="MS")

fig = go.Figure()
fig.add_trace(go.Scatter(x=monthly["fecha"], y=monthly["total"], mode="lines",
                         name="Histórico", line={"color":"#58a6ff","width":1.5}))
# Banda de confianza
fig.add_trace(go.Scatter(x=list(future_dates)+list(future_dates[::-1]),
                         y=list(y_future_high)+list(y_future_low[::-1]),
                         fill="toself", fillcolor="rgba(241,136,62,0.18)",
                         line={"color":"rgba(0,0,0,0)"}, name="IC 95%", hoverinfo="skip"))
fig.add_trace(go.Scatter(x=future_dates, y=y_future, mode="lines+markers",
                         name="Pronóstico 12m", line={"color":"#f0883e","width":2.5,"dash":"dot"}))
fig.update_layout(title="Pronóstico de denuncias mensuales — próximos 12 meses",
                  xaxis_title="", yaxis_title="Denuncias / mes",
                  height=420, plot_bgcolor="white")
fig.show()

print(f"📈 Proyección 12 meses adelante: total esperado ≈ {y_future.sum():.0f} denuncias")
print(f"   IC 95%: [{y_future_low.sum():.0f}, {y_future_high.sum():.0f}]")
print(f"\n⚠️ Tomar este pronóstico con pinzas: hay gap 2020-2022 sin datos.")

## 1.1 Mini-KPIs de tendencia por categoría top

Para cada una de las 4 categorías más frecuentes, calculo si la tendencia reciente (último año vs. promedio histórico) está subiendo, estable o bajando.

In [ ]:
ult_año = df["año"].max()
for cat in df["categoria_label"].value_counts().head(4).index:
    sub = df[df["categoria_label"] == cat]
    media_hist = len(sub) / sub["año"].nunique()
    ult = len(sub[sub["año"] == ult_año])
    delta = (ult - media_hist) / media_hist * 100
    flecha = "📈" if delta > 15 else ("📉" if delta < -15 else "➡️")
    print(f"  {flecha} {cat:35s} {ult_año}: {ult:4d}  (vs. media histórica {media_hist:4.0f}, {delta:+.0f}%)")

# 2️⃣ NLP — términos frecuentes en descripciones

El histórico 2010-2019 tiene un campo `descripcion_libre` con texto de la denuncia. Esto es **oro** para análisis lingüístico. Hago un baseline simple: frecuencia de términos por categoría, con stopwords.

In [ ]:
STOPWORDS_ES = {
    "de","la","el","en","y","a","que","los","las","un","una","se","por","con",
    "para","del","al","es","o","como","no","su","sus","este","esta","estos",
    "ese","esa","esos","esas","si","sí","pero","o","u","e","ni","lo","le",
    "les","nos","yo","tu","tú","él","ella","ellos","ellas","nosotros",
    "vosotros","ustedes","hay","han","ha","he","hemos","habia","sin","sobre",
    "entre","hasta","desde","durante","ante","tras","contra","según","mediante",
    "muy","más","mas","menos","también","tambien","ya","todo","toda","todos",
    "todas","otro","otra","otros","otras","mismo","misma","mi","tu","un","una",
    "unos","unas","está","esta","están","estan","ser","sea","sean","era","fue",
    "fueron","hace","hacen","haciendo","sólo","solo","así","asi","aún","aun",
}

import re
def tokenize(t):
    if pd.isna(t):
        return []
    t = str(t).lower()
    t = re.sub(r"[^a-záéíóúüñ\s]", " ", t)
    return [w for w in t.split() if len(w) >= 4 and w not in STOPWORDS_ES]

# Top 15 términos globales
from collections import Counter
tokens_all = []
for d in df["descripcion_libre"].dropna():
    tokens_all.extend(tokenize(d))

top_global = Counter(tokens_all).most_common(15)
print(f"Total tokens analizados: {len(tokens_all):,}")
print(f"\n🔤 Top 15 términos globales:")
for w, n in top_global:
    print(f"   {w:20s} {n:5d}")

In [ ]:
# Bar chart de top 15 global
palabras, conteos = zip(*top_global)
fig = go.Figure(go.Bar(x=conteos, y=palabras, orientation="h",
                       marker_color="#1abc9c"))
fig.update_layout(title="Top 15 términos en descripciones libres (todo el histórico)",
                  yaxis={"categoryorder": "total ascending"},
                  height=420, plot_bgcolor="white")
fig.show()

In [ ]:
# Top términos por categoría
print("🔤 Top 8 términos por categoría top:\n")
for cat in df["categoria_label"].value_counts().head(5).index:
    tokens_cat = []
    for d in df[df["categoria_label"] == cat]["descripcion_libre"].dropna():
        tokens_cat.extend(tokenize(d))
    if tokens_cat:
        top_cat = Counter(tokens_cat).most_common(8)
        top_str = ", ".join(f"{w} ({n})" for w, n in top_cat)
        print(f"  📍 {cat}")
        print(f"     {top_str}\n")

# 3️⃣ Sistema de alertas

Detección de meses anómalos recientes y conteo de urgentes pendientes.

In [ ]:
# Anomalías del último año
monthly_z = monthly.copy()
monthly_z["zscore"] = (monthly_z["total"] - monthly_z["total"].mean()) / monthly_z["total"].std()
monthly_z["alerta"] = monthly_z["zscore"].abs() > 2.0

ult_año_data = monthly_z[monthly_z["fecha"].dt.year == ult_año]
alertas_recientes = ult_año_data[ult_año_data["alerta"]]

print(f"🚨 Alertas detectadas en {ult_año}: {len(alertas_recientes)}")
for _, r in alertas_recientes.iterrows():
    icono = "🔺" if r["zscore"] > 0 else "🔻"
    print(f"   {icono} {r['fecha'].strftime('%Y-%m')}: {int(r['total'])} denuncias (z={r['zscore']:+.2f})")

if not len(alertas_recientes):
    print("   (sin meses anómalos en el último año)")

# Urgentes en últimos 30 días
urg_30d = pd.to_numeric(
    df[df["timestamp"] >= pd.Timestamp.now() - pd.Timedelta(days=30)]["urgencia"],
    errors="coerce"
).fillna(0).sum()
print(f"\n🆘 Denuncias urgentes en últimos 30 días: {int(urg_30d)}")
print("   (esperá datos del formulario para que este indicador se active)")

# Denuncias nuevas del formulario
n_form = (df["fuente"] == "formulario").sum() if "fuente" in df.columns else 0
print(f"\n📥 Denuncias capturadas por formulario: {n_form}")

# 4️⃣ Clustering territorial (K-Means)

Cada departamento se representa como un vector de 9 dimensiones (% de cada categoría). K-Means agrupa departamentos con perfiles similares.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Pivot: filas=depto, columnas=categoría, valores=% del total del depto
pivot = (df.pivot_table(index="departamento", columns="categoria_label",
                        values="id_denuncia", aggfunc="count", fill_value=0))
pivot_norm = pivot.div(pivot.sum(axis=1), axis=0)

# K-Means k=4
X = StandardScaler().fit_transform(pivot_norm.values)
km = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X)
pivot_norm["cluster"] = km.labels_

print("📍 Asignación de clusters:")
for c in sorted(pivot_norm["cluster"].unique()):
    deptos = pivot_norm[pivot_norm["cluster"] == c].index.tolist()
    print(f"\n  Cluster {c} ({len(deptos)} deptos): {', '.join(deptos)}")
    # Categoría dominante de cada cluster (la del % más alto promedio)
    mean_profile = pivot_norm[pivot_norm["cluster"]==c].drop("cluster", axis=1).mean()
    top3 = mean_profile.nlargest(3)
    print(f"     Perfil top: {', '.join(f'{k} ({v*100:.0f}%)' for k,v in top3.items())}")

In [ ]:
# Mapa coloreado por cluster
DEPT_COORDS = {
    "Artigas":(-30.40,-56.47),"Canelones":(-34.52,-56.28),"Cerro Largo":(-32.37,-54.18),
    "Colonia":(-34.46,-57.84),"Durazno":(-33.38,-56.52),"Flores":(-33.55,-56.90),
    "Florida":(-34.10,-56.21),"Lavalleja":(-34.38,-55.24),"Maldonado":(-34.91,-54.96),
    "Montevideo":(-34.90,-56.16),"Paysandú":(-32.32,-58.08),"Río Negro":(-33.13,-58.31),
    "Rivera":(-30.91,-55.55),"Rocha":(-34.48,-54.34),"Salto":(-31.39,-57.97),
    "San José":(-34.34,-56.71),"Soriano":(-33.45,-58.04),"Tacuarembó":(-31.71,-55.99),
    "Treinta y Tres":(-33.23,-54.38),
}
CLUSTER_COLORS = ["#1abc9c", "#f0883e", "#58a6ff", "#d2a8ff"]

data_map = pivot_norm[["cluster"]].reset_index()
data_map["lat"] = data_map["departamento"].map(lambda d: DEPT_COORDS[d][0])
data_map["lon"] = data_map["departamento"].map(lambda d: DEPT_COORDS[d][1])
data_map["color"] = data_map["cluster"].map(lambda c: CLUSTER_COLORS[c])

fig = go.Figure(go.Scattermapbox(
    lat=data_map["lat"], lon=data_map["lon"], mode="markers+text",
    marker={"size":22, "color":data_map["color"], "opacity":0.85},
    text=data_map["departamento"], textposition="top center",
    hovertemplate="<b>%{text}</b><br>Cluster %{customdata}<extra></extra>",
    customdata=data_map["cluster"],
))
fig.update_layout(
    mapbox={"style":"carto-positron","center":{"lat":-32.7,"lon":-56.5},"zoom":5.3},
    title="Clusters territoriales de Uruguay (K-Means, k=4)",
    height=520, margin={"t":40,"r":0,"b":0,"l":0})
fig.show()

---
## ✅ Cierre

- **Pronóstico**: bajo nuestra parametrización, los próximos 12 meses traerían un volumen comparable al 2023. Con la limitación clara del gap 2020-2022.
- **NLP**: los términos frecuentes confirman categorías (cianobacterias, vertidos, ruidos, residuos). Sirve para detectar nuevos motivos no catalogados.
- **Alertas**: la maquinaria está lista pero necesita datos del formulario para llenarse.
- **Clustering**: emerge un patrón geográfico claro. Los departamentos costeros forman un grupo, el interior productivo otro, etc.

**Próximo notebook**: `05_nueva_denuncia.ipynb` — documentación pedagógica del formulario del tablero.